## GW190425 with Bilby

Tutorial to demonstrate running parameter estimation on GW190425.

This example estimates all 17 parameters of the binary neutron star system using
commonly used prior distributions.
We use the relative binning likelihood to reduce the computational cost.
This should take around ten minutes to run on a single CPU core.
We expect the data has been pre-downloaded,
however this can be done by running the following code in a cell below.

```python
! mkdir -p data
! cd data
! wget https://gwosc.org/eventapi/html/GWTC-2.1-confident/GW190425/v3/L-L1_GWOSC_16KHZ_R1-1240213455-4096.gwf
! wget https://gwosc.org/eventapi/html/GWTC-2.1-confident/GW190425/v3/V-V1_GWOSC_16KHZ_R1-1240213455-4096.gwf
! cd -
```

First we need to include some boilerplate to avoid serious
performance bottlenecks using `lal` in a `Jupyter` notebook.

In [ ]:
import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal
lal.swig_redirect_standard_output_error(False)

Next, we will import the packages/classes that we are going to use during this example.
I've explicitly spelled out the `bilby` imports so that we can see which aspects of the code we need.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from bilby import run_sampler
from bilby.core.prior import Constraint, Uniform, Sine, Cosine, PowerLaw
from bilby.core.utils import check_directory_exists_and_if_not_mkdir
from bilby.gw.conversion import convert_to_lal_binary_neutron_star_parameters, generate_all_bns_parameters
from bilby.gw.detector import get_empty_interferometer, InterferometerList, PowerSpectralDensity
from bilby.gw.likelihood import RelativeBinningGravitationalWaveTransient
from bilby.gw.prior import BNSPriorDict, UniformInComponentsChirpMass, UniformInComponentsMassRatio
from bilby.gw.result import CBCResult
from bilby.gw.source import lal_binary_neutron_star_relative_binning
from bilby.gw.waveform_generator import WaveformGenerator
from gwpy.timeseries import TimeSeries

Here we define some boilerplate to set up the analysis.

Most of these should be fairly clear.
The `roll_off` is a parameter that determines the windowing function we will use.

In [ ]:
outdir = "outdir"
label = "GW190425"
trigger_time = 1240215503.0
detectors = ["L1", "V1"]
maximum_frequency = 512
minimum_frequency = 20
roll_off = 0.4
duration = 128
post_trigger_duration = 2
end_time = trigger_time + post_trigger_duration
start_time = end_time - duration

### Load the data

Here we load in the data from the downloaded `.gwf` files using `gwpy`
and use it to set the data in the `bilby.gw.detector.interferometer.Interferometer`
objects.

We then read more of the data to estimate the noise power spectral densities.
We will esimate these using a median avergae of 16 second segments ending before the analysis segments.

In [ ]:
psd_duration = 1024
psd_start_time = start_time - psd_duration
psd_end_time = start_time

data_files = dict(
    L1="data/L-L1_GWOSC_16KHZ_R1-1240213455-4096.gwf",
    V1="data/V-V1_GWOSC_16KHZ_R1-1240213455-4096.gwf",
)

ifo_list = InterferometerList([])
for det in detectors:
    ifo = get_empty_interferometer(det)
    data = TimeSeries.read(data_files[det], channel=f"{det}:GWOSC-16KHZ_R1_STRAIN", start=start_time, end=end_time)
    ifo.strain_data.set_from_gwpy_timeseries(data)

    psd_data = TimeSeries.read(data_files[det], channel=f"{det}:GWOSC-16KHZ_R1_STRAIN", start=psd_start_time, end=psd_end_time)
    psd_alpha = 2 * roll_off / 16
    psd = psd_data.psd(
        fftlength=16, overlap=0, window=("tukey", psd_alpha), method="median"
    )
    # save the PSD so we can use it again later
    psd.write(f"data/{det}-psd.dat", format="txt")
    ifo.power_spectral_density = PowerSpectralDensity(
        frequency_array=psd.frequencies.value, psd_array=psd.value
    )
    ifo.maximum_frequency = maximum_frequency
    ifo.minimum_frequency = minimum_frequency
    ifo_list.append(ifo)

check_directory_exists_and_if_not_mkdir(outdir)
ifo_list.plot_data(outdir=outdir, label=label)

This cell saved plots into the `outdir` directory showing the frequency-domain data in each detector.
Take a look at them.

### Define the prior

There are some notable things in this prior definition:

- The prior is defined over chirp mass and mass ratio.
  This makes the sampling problem easier as the chirp mass is very well measured.
  We also use the `UniformInComponents{ChirpMass,MassRatio}` priors which, when
  used together, give a prior which is equivalent to setting uniform priors in
  mass_1 and mass_2 with constraints in chirp mass and mass ratio.
- We constrain the two component masses to lie within [1, 2.5] which is approximately
  the expected masses allowed for neutron stars.
- We define two priors `zenith` and `azimuth` rather than `ra` and `dec` to define
  the position on the sky. This defines the sky location relative to the vector
  connecting two interferometers. This is a much more natural basis for the problem
  as cos(zenith) is proportional to the time delay between the two instruments.
- We define a prior on `L1_time` which is the merger time at LIGO Livingston,
  again this is a more natural parameter as we measure the time at Livingston
  much better than the time the signal reaches the geocenter.

In [ ]:
priors = BNSPriorDict(dict(
    chirp_mass=UniformInComponentsChirpMass(name='chirp_mass', minimum=1.485, maximum=1.49, unit='$M_{\\odot}$'),
    mass_ratio=UniformInComponentsMassRatio(name='mass_ratio', minimum=0.125, maximum=1),
    mass_1=Constraint(name='mass_1', minimum=1, maximum=2.5),
    mass_2=Constraint(name='mass_2', minimum=1, maximum=2.5),
    a_1=Uniform(name='a_1', minimum=0, maximum=0.05),
    a_2=Uniform(name='a_2', minimum=0, maximum=0.05),
    tilt_1=Sine(name='tilt_1'),
    tilt_2=Sine(name='tilt_2'),
    phi_12=Uniform(name='phi_12', minimum=0, maximum=2 * np.pi, boundary='periodic'),
    phi_jl=Uniform(name='phi_jl', minimum=0, maximum=2 * np.pi, boundary='periodic'),
    luminosity_distance=PowerLaw(alpha=2, name='luminosity_distance', minimum=10, maximum=1000, unit='Mpc', latex_label='$d_L$'),
    zenith=Sine(name='zenith', latex_label="$\\epsilon$"),
    azimuth=Uniform(name='azimuth', minimum=0, maximum=2 * np.pi, boundary='periodic', latex_label="$\\alpha$"),
    theta_jn=Sine(name='theta_jn'),
    psi=Uniform(name='psi', minimum=0, maximum=np.pi / 2, boundary='periodic'),
    phase=Uniform(name='phase', minimum=0, maximum=2 * np.pi, boundary='periodic'),
    lambda_1=Uniform(name='lambda_1', minimum=0, maximum=5000),
    lambda_2=Uniform(name='lambda_2', minimum=0, maximum=5000),
    L1_time=Uniform(trigger_time - 0.05, trigger_time + 0.05, name="geocent_time"),
    fiducial=0,
))

### Set the fiducial parameters

Since we are using the relative binning likelihood,
we will specify fiducial parameters for the approximation.
It is possible to have the fiducial parameters found automatically using `scipy` optimization,
however, since we have access to a good estimate for this event, we may as well use them.

The fiducial parameters are taken to be the max likelihood sample from the
posterior sample release of LIGO-Virgo
https://www.gw-openscience.org/eventapi/html/O3_Discovery_Papers/GW190425/.

**Note**: the fiducial parameters are not specific using the same parameterization as the prior.
In earlier versions of `bilby` this would have lead to issues, but not anymore!

In [ ]:
fiducial_parameters = {
    "a_1": 0.018,
    "a_2": 0.016,
    "chirp_mass": 1.48658,
    "dec": 0.438,
    "geocent_time": 1240215503.039,
    "lambda_1": 446.941,
    "lambda_2": 43.386,
    "luminosity_distance": 206.751,
    "mass_ratio": 0.8955,
    "phase": 3.0136566567608765,
    "phi_12": 4.319,
    "phi_jl": 5.07,
    "psi": 0.281,
    "ra": 4.2,
    "theta_jn": 0.185,
    "tilt_1": 0.879,
    "tilt_2": 0.514,
}

### Create the waveform generator

The waveform generator defines how the waveform should be calculated.
The default `WaveformGenerator` can be used with a range of source functions.
In this case we will use the a waveform model to generate waveforms for binaries
containing at least one neutron star using `lalsimulation`.
We also pass a `parameter_conversion` which enables us to evaluate the waveform with
any (consistent) set of parameters, e.g., component masses or chirp mass and mass ratio.
We are using the `IMRPhenomXP_NRTidalv3` waveform model, which includes tidal effects
tuned to numerical relativity.

In [ ]:
waveform_generator = WaveformGenerator(
    frequency_domain_source_model=lal_binary_neutron_star_relative_binning,
    parameter_conversion=convert_to_lal_binary_neutron_star_parameters,
    waveform_arguments={
        "waveform_approximant": "IMRPhenomXP_NRTidalv3",
        "reference_frequency": 20,
    },
)

### Construct the likelihood

We can now assemble the likelihood using the objects defined above.

Notes:
- We are using phase and distance marginalization. Phase marginalization is
  strictly not valid for this waveform model, but it is still a good approximation.
- We declare the reference frame for the sky location and the time reference,
  these should match the parameters specified in the prior.

In [ ]:
likelihood = RelativeBinningGravitationalWaveTransient(
    ifo_list,
    waveform_generator,
    priors=priors,
    time_marginalization=False,
    phase_marginalization=True,
    distance_marginalization=True,
    fiducial_parameters=fiducial_parameters,
    reference_frame="H1L1",
    time_reference="L1",
)

### Run the sampler

Now we're ready to start sampling.
We'll use `dynesty` which is default sampler and the one most widely used for
CBC inference.
The `acceptance-walk` method is a custom proposal distribution for `dynesty` implemented
directly in `bilby`.

We specify a `conversion_function` which will generate various useful parameters in post-processing.

In [ ]:
result = run_sampler(
    likelihood,
    priors,
    sampler="dynesty",
    outdir=outdir,
    label=label,
    nlive=500,
    sample="acceptance-walk",
    naccept=5,
    check_point_delta_t=600,
    check_point_plot=True,
    npool=1,
    save="hdf5",
    resume=False,
    clean=True,
    conversion_function=generate_all_bns_parameters,
    result_class=CBCResult,
)

### Plotting

Finally let's make some plots!

We'll make some small corner plots.
Note that we didn't define priors for (almost) all of the parameters shown below.
These are available due to the `conversion_function above`.

In [ ]:
_ = result.plot_corner(save=False, parameters=["mass_1_source", "mass_2_source", "chi_eff", "chi_p"])
plt.show()
plt.close()


_ = result.plot_corner(save=False, parameters=["ra", "dec", "luminosity_distance"])
plt.show()
plt.close()